## Bước 1: Load dữ liệu

In [1]:
import pandas as pd

FILE_NAME = r'C:\Users\Admin\Documents\NEW_JOURNEY\PROJECT_3\online_retail_II.csv'  # sua duong dan neu can

df = pd.read_csv(FILE_NAME, encoding='ISO-8859-1')
print(f"So dong ban dau: {len(df)}")

So dong ban dau: 1067371


## Bước 2: Loại bỏ dòng thiếu Customer ID

In [2]:
df = df.dropna(subset=['Customer ID'])
print(f"Sau khi loai thieu Customer ID: {len(df)}")

Sau khi loai thieu Customer ID: 824364


## Bước 3: Loại bỏ bút toán điều chỉnh nợ xấu (Invoice bắt đầu bằng 'A')

#Invoice: mã đơn hàng

#StockCode/Description: mã và tên sản phẩm

#Quantity, Price: số lượng và đơn giá của sản phẩm đó trong đơn hàng

#Customer ID: mã khách hàng (ai đã mua)

#InvoiceDate: thời điểm mua

In [3]:
df = df[~df['Invoice'].astype(str).str.startswith('A')]
print(f"Sau khi loai but toan dieu chinh no xau: {len(df)}")

Sau khi loai but toan dieu chinh no xau: 824364


Do khi loại bỏ các dòng thiếu customer ID, các dòng đó cũng có nợ xấu (A) nên cũng bị loại bỏ đồng thời

## Bước 4: Loại bỏ dòng trùng lặp hoàn toàn

In [4]:
before = len(df)
df = df.drop_duplicates()
print(f"Loai {before - len(df)} dong trung lap, con lai {len(df)} dong")

Loai 26479 dong trung lap, con lai 797885 dong


## Bước 5: Đánh dấu đơn hàng bị hủy (Invoice bắt đầu bằng 'C')

In [5]:
df['is_cancelled'] = df['Invoice'].astype(str).str.startswith('C')
print(f"So dong la don huy: {df['is_cancelled'].sum()}")

So dong la don huy: 18390


## Bước 6: Tính giá trị từng dòng và xác định ngày mốc (snapshot date)

In [6]:
df['line_value'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Ngay moc (snapshot date) dung de tinh Recency: {snapshot_date}")

Ngay moc (snapshot date) dung de tinh Recency: 2011-12-10 12:50:00


Monetary = tổng số tiền đã chi (đã trừ hàng hủy/trả) → số lớn = tốt

Recency = đã bao nhiêu ngày KHÔNG mua (tính từ lần mua cuối tới "mốc hôm nay" giả định) → số nhỏ = tốt (mới mua gần đây)

Frequency = đã mua bao nhiêu LẦN (bao nhiêu đơn hàng khác nhau) → số lớn = tốt



## Bước 7: Tính Monetary (trên toàn bộ dòng, kể cả đơn hủy)

In [ ]:
#tính monetary, ta cần tính tổng giá trị mua hàng của mỗi khách hàng. Ta sẽ nhóm dữ liệu theo 'Customer ID' và tính tổng giá trị của cột 'line_value' cho mỗi khách hàng.

monetary = df.groupby('Customer ID')['line_value'].sum().reset_index()
monetary.columns = ['Customer ID', 'Monetary']

print(monetary.head())

   Customer ID  Monetary
0      12346.0    -51.74
1      12347.0   4921.53
2      12348.0   2019.40
3      12349.0   4404.54
4      12350.0    334.40


## Bước 8: Tính Recency và Frequency (chỉ trên đơn hàng THẬT, không tính đơn hủy)

In [ ]:
# tính Recency, ta cần xác định ngày gần nhất mà khách hàng mua hàng. Ta sẽ nhóm dữ liệu theo 'Customer ID' và lấy ngày mua hàng gần nhất của mỗi khách hàng, sau đó tính số ngày kể từ ngày đó đến ngày snapshot_date.
# tính frequency, ta cần đếm số lần mua hàng của mỗi khách hàng. Ta sẽ nhóm dữ liệu theo 'Customer ID' và đếm số lượng hóa đơn (Invoice) duy nhất mà mỗi khách hàng đã thực hiện.
df_valid = df[~df['is_cancelled']]

rf = df_valid.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
).reset_index()

print(rf.head())

   Customer ID  Recency  Frequency
0      12346.0      326         12
1      12347.0        2          8
2      12348.0       75          5
3      12349.0       19          4
4      12350.0      310          1


## Bước 9: Gộp lại thành bảng RFM hoàn chỉnh, kiểm tra khách chỉ có đơn hủy

In [9]:
rfm = monetary.merge(rf, on='Customer ID', how='left')

missing_rf = rfm['Recency'].isna().sum()
print(f"So khach hang CHI co don huy (khong co don that): {missing_rf}")

if missing_rf > 0:
    rfm = rfm.dropna(subset=['Recency'])
    print(f"Da loai {missing_rf} khach hang nay")

print(f"\nTong so khach hang con lai de phan tich: {len(rfm)}")
print(rfm.head(10))

So khach hang CHI co don huy (khong co don that): 61
Da loai 61 khach hang nay

Tong so khach hang con lai de phan tich: 5881
   Customer ID  Monetary  Recency  Frequency
0      12346.0    -51.74    326.0       12.0
1      12347.0   4921.53      2.0        8.0
2      12348.0   2019.40     75.0        5.0
3      12349.0   4404.54     19.0        4.0
4      12350.0    334.40    310.0        1.0
5      12351.0    300.93    375.0        1.0
6      12352.0   1889.21     36.0       10.0
7      12353.0    406.76    204.0        2.0
8      12354.0   1079.40    232.0        1.0
9      12355.0    947.61    214.0        2.0


In [10]:
kh_am = rfm[rfm['Monetary'] < 0]
print(f"So khach hang co Monetary am: {len(kh_am)}")

So khach hang co Monetary am: 23


Monetary = tổng số tiền đã chi (đã trừ hàng hủy/trả) → số lớn = tốt

Recency = đã bao nhiêu ngày KHÔNG mua (tính từ lần mua cuối tới "mốc hôm nay" giả định) → số nhỏ = tốt (mới mua gần đây)

Frequency = đã mua bao nhiêu LẦN (bao nhiêu đơn hàng khác nhau) → số lớn = tốt



In [11]:
# Xuat du lieu giao dich chi tiet (df) ra CSV de nap vao bang transactions
df_export = df[['Invoice', 'StockCode', 'Description', 'Quantity',
                 'InvoiceDate', 'Price', 'Customer ID', 'Country',
                 'is_cancelled', 'line_value']].copy()

df_export['is_cancelled'] = df_export['is_cancelled'].astype(int)  # True/False -> 1/0

df_export.to_csv('transactions_clean.csv', index=False, encoding='utf-8-sig')
print(f"Da luu {len(df_export)} dong giao dich vao transactions_clean.csv")

Da luu 797885 dong giao dich vao transactions_clean.csv


In [11]:
rfm.to_csv('rfm_customers.csv', index=False, encoding='utf-8-sig')
print("Da luu vao rfm_customers.csv")

Da luu vao rfm_customers.csv
